# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/xtallet/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/xtallet/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '158767'. Skipping!
Property 'summary' already exists in node 'a417b7'. Skipping!
Property 'summary' already exists in node '642f4f'. Skipping!
Property 'summary' already exists in node 'a32685'. Skipping!
Property 'summary' already exists in node 'e5aaa8'. Skipping!
Property 'summary' already exists in node 'df01dd'. Skipping!
Property 'summary' already exists in node 'cfd04c'. Skipping!
Property 'summary' already exists in node '383026'. Skipping!
Property 'summary' already exists in node 'fb9baa'. Skipping!
Property 'summary' already exists in node '06331a'. Skipping!
Property 'summary' already exists in node '9a0345'. Skipping!
Property 'summary' already exists in node '834042'. Skipping!
Property 'summary' already exists in node 'fb18c9'. Skipping!
Property 'summary' already exists in node '7baa21'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '158767'. Skipping!
Property 'summary_embedding' already exists in node 'a417b7'. Skipping!
Property 'summary_embedding' already exists in node 'a32685'. Skipping!
Property 'summary_embedding' already exists in node 'df01dd'. Skipping!
Property 'summary_embedding' already exists in node '7baa21'. Skipping!
Property 'summary_embedding' already exists in node '383026'. Skipping!
Property 'summary_embedding' already exists in node 'e5aaa8'. Skipping!
Property 'summary_embedding' already exists in node 'cfd04c'. Skipping!
Property 'summary_embedding' already exists in node 'fb9baa'. Skipping!
Property 'summary_embedding' already exists in node 'fb18c9'. Skipping!
Property 'summary_embedding' already exists in node '9a0345'. Skipping!
Property 'summary_embedding' already exists in node '642f4f'. Skipping!
Property 'summary_embedding' already exists in node '06331a'. Skipping!
Property 'summary_embedding' already exists in node '834042'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5), # XTALLET Notes - Generates simple and direct questions
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25), # XTALLET Notes - Generates more complex and abstract questions
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25), # XTALLET Notes - Generates specific and complex questions
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

- SingleHopSpecificQuerySynthesizer :
  It generates simple and direct questions that can be answered by looking at a single piece of information in the text.<br>
  It makes easy and direct questions.<br>
  These questions will represent the 50% of the total list of queries.

- MultiHopAbstractQuerySynthesizer :
  Generates more complex and general questions that require combining information from different parts of the text and thinking more abstractly.<br>
  It makes broad and challenging questions.<br>
  These questions will represent the 25% of the total list of queries.

- MultiHopSpecificQuerySynthesizer : 
  Generates specific but complex questions that require searching and connecting information from multiple parts of the text to give a concrete answer.<br>
  It makes detailed questions that require gathering information from different parts of the text.<br>
  These questions will represent the 25% of the total list of queries.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Whay are Academic Years important?,"[Chapter 1 Academic Years, Academic Calendars,...",Academic Years are important because every eli...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding ac...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) pertains to the minimum number...,single_hop_specifc_query_synthesizer
2,What information does Volume 8 provide regardi...,[Inclusion of Clinical Work in a Standard Term...,Volume 8 explains that clinical work conducted...,single_hop_specifc_query_synthesizer
3,FWS is it a payment period?,[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
4,What is the Pell Grannt?,[both the credit or clock hours and the weeks ...,The Pell Grant is a type of federal financial ...,single_hop_specifc_query_synthesizer
5,How do separate academic years for different p...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that a school may define ...,multi_hop_abstract_query_synthesizer
6,How do participation requirements for practicu...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Participation in practicum or clinical experie...,multi_hop_abstract_query_synthesizer
7,"How do non-term characteristics, such as cours...",[<1-hop>\n\nInclusion of Clinical Work in a St...,"Non-term characteristics, like courses that do...",multi_hop_abstract_query_synthesizer
8,How does Volume 8 explain the impact of accele...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 details that accelerated progression ...,multi_hop_specific_query_synthesizer
9,Volume 2 and Volume 7 how do they relate to di...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 2 explains that the scheduled payment p...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'c5e32c'. Skipping!
Property 'summary' already exists in node '000ab5'. Skipping!
Property 'summary' already exists in node 'e82c24'. Skipping!
Property 'summary' already exists in node '71cf39'. Skipping!
Property 'summary' already exists in node '56aae3'. Skipping!
Property 'summary' already exists in node '77ea68'. Skipping!
Property 'summary' already exists in node 'f84874'. Skipping!
Property 'summary' already exists in node '2e5a95'. Skipping!
Property 'summary' already exists in node '2085a0'. Skipping!
Property 'summary' already exists in node 'a889f2'. Skipping!
Property 'summary' already exists in node '1e6b6d'. Skipping!
Property 'summary' already exists in node 'c1bd7c'. Skipping!
Property 'summary' already exists in node 'e620ff'. Skipping!
Property 'summary' already exists in node '0980ac'. Skipping!


...

Property 'summary_embedding' already exists in node 'c5e32c'. Skipping!
Property 'summary_embedding' already exists in node 'e82c24'. Skipping!
Property 'summary_embedding' already exists in node '77ea68'. Skipping!
Property 'summary_embedding' already exists in node '000ab5'. Skipping!
Property 'summary_embedding' already exists in node '56aae3'. Skipping!
Property 'summary_embedding' already exists in node '2085a0'. Skipping!
Property 'summary_embedding' already exists in node '2e5a95'. Skipping!
Property 'summary_embedding' already exists in node '71cf39'. Skipping!
Property 'summary_embedding' already exists in node 'c1bd7c'. Skipping!
Property 'summary_embedding' already exists in node 'e620ff'. Skipping!
Property 'summary_embedding' already exists in node '0980ac'. Skipping!
Property 'summary_embedding' already exists in node 'a889f2'. Skipping!
Property 'summary_embedding' already exists in node '1e6b6d'. Skipping!
Property 'summary_embedding' already exists in node 'f84874'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What role does the Department play in defining...,"[Chapter 1 Academic Years, Academic Calendars,...",The Department oversees the definition of acad...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding th...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) specifies the minimum requirem...,single_hop_specifc_query_synthesizer
2,What does Volume 8 specify regarding clinical ...,[Inclusion of Clinical Work in a Standard Term...,Volume 8 states that clinical work conducted o...,single_hop_specifc_query_synthesizer
3,How does Title IV regulation relate to payment...,[Non-Term Characteristics A program that measu...,"According to the context, Title IV program dis...",single_hop_specifc_query_synthesizer
4,How does accelerated progression in clock-hour...,[<1-hop>\n\nboth the credit or clock hours and...,Accelerated progression in clock-hour or non-t...,multi_hop_abstract_query_synthesizer
5,How do the disbursement timing requirements fo...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement timing requirements for sched...,multi_hop_abstract_query_synthesizer
6,How do the disbursement timing requirements fo...,[<1-hop>\n\nboth the credit or clock hours and...,Disbursement timing for federal student aid va...,multi_hop_abstract_query_synthesizer
7,Hwo do payment perods relate to 34 CFR 668.3(a...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Payment periods are defined within the context...,multi_hop_abstract_query_synthesizer
8,How do Volume 2 and Volume 8 relate to the inc...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Volume 2 discusses the requirements for academ...,multi_hop_specific_query_synthesizer
9,whats volume 8 and volume 7 about?,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 discusses the timing of disbursements...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent PLUS Loans)  \n- Subsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)  \n- Unsubsidized Federal Stafford Loans (under the FFEL Program before July 1, 2010)  \n- Federal SLS Loans (under the FFEL Program before July 1, 2010)\n\nAdditionally, Direct PLUS Loans and Direct Unsubsidized Loans can replace the SAI and cover unmet need, while Direct Subsidized Loans are available only to undergraduate students. Graduate or professional students are eligible for Direct Unsubsidized Loans but not for Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

##### ✅ Answer:

- QA_EVALUATOR :
  Evaluates the correctness of the model's answer. It checks wheter the generated response accurately answers the user's question based on the reference answer (the answer from synthetic dataset).

- LABELED_HELPFULNESS_EVALUATOR :
  Evaluates the helpfulness of the response. It determines if the answer is useful and supportive to the user, taking into account the correct reference answer (the answer from synthetic dataset).

- EMPATHY_EVALUATOR :
  Evaluates the empathy in the response. It checks whether the answer is empathetic and makes the user feel heard and understood.
   

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'yellow-expert-58' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/b02303d2-7182-4bcf-a2e9-b7c30b9f6ca2/compare?selectedSessions=3f91ee85-8d35-4ee8-b527-6d750fde8e58




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendix A and Appendix B provide guida...,I don't know.,None,Appendix A offers guidance on the disbursement...,0,0,0,2.313489,c129ff14-f501-40c1-97ff-dd4c38978119,76e0260c-a0e3-4c04-b871-98f0246b1871
1,"In volume 2 and volume 7, how does the academi...",Based on the provided context from volumes 2 a...,None,"According to volume 2, the disbursement of fed...",1,1,0,8.009226,200ebfc3-0bc5-41ec-862d-6a8f5be80f7d,fc17e207-12a1-4ddb-80e3-949894b05093
2,whats volume 8 and volume 7 about?,Based on the provided context:\n\n- Volume 7 i...,None,Volume 8 discusses the timing of disbursements...,0,0,0,1.795341,f0d16cb8-1dc8-446d-83e0-87712bc241f2,cf5d36a7-7a98-4a6e-84a0-548a49f0b19b
3,How do Volume 2 and Volume 8 relate to the inc...,I don't know.,None,Volume 2 discusses the requirements for academ...,0,0,0,1.826557,13f56a47-af20-4d4c-918c-befa8470b704,903bc941-e5de-43d8-af77-da68b4ad23ae
4,Hwo do payment perods relate to 34 CFR 668.3(a...,I don't know.,None,Payment periods are defined within the context...,0,0,0,1.130355,6b508c69-5ff0-46ae-b9c1-6577d43fd2a8,2401a9b0-0722-4583-a8ab-93b1d3ee31f6
5,How do the disbursement timing requirements fo...,Based on the provided context:\n\n**Disburseme...,None,Disbursement timing for federal student aid va...,1,1,0,12.950859,9118dee9-e36e-4d56-891a-6b0a943be861,650de9ff-5721-4b86-a8ba-9fb71708d352
6,How do the disbursement timing requirements fo...,"Based on the provided context, the disbursemen...",None,The disbursement timing requirements for sched...,1,1,0,8.524249,59f97a86-f91a-439b-b41f-ff1b307984d9,b90543ce-4f0d-47c8-abc5-a33d0b158ff3
7,How does accelerated progression in clock-hour...,"Based on the provided context, accelerated pro...",None,Accelerated progression in clock-hour or non-t...,1,1,0,14.103013,6de43c54-5e0d-47d7-9ea3-28956d5ceb80,c251290e-a529-491d-923f-1b37f5ebc2df
8,How does Title IV regulation relate to payment...,Title IV regulations require that disbursement...,None,"According to the context, Title IV program dis...",1,1,0,3.295089,c58a4368-d396-4d62-86a9-e2461844f1ad,edd69b27-919c-48e5-8611-84d8d67ff5cd
9,What does Volume 8 specify regarding clinical ...,"Volume 8, Chapter 3 provides additional guidan...",None,Volume 8 states that clinical work conducted o...,0,0,0,1.429825,6c937831-dee8-4803-aac2-8e782b81d5fd,dbd42eef-b74c-421b-b8d7-3b19e283e77e


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Modifying the chunk size can signigicantly impact on :
 - Retrieval Quality :<br> 
   Smaller chunks provide more precise and focused information   retrieval, but may miss broader context.<br>
   Larger chunks : Capture more comprehensive context but may include irrelevant information that dilutes the answer quality.

 - Context Window Efficiency :<br>
   Smaller chunks fit better within the LLM's context window, allowing more relevant chunks to be included in the prompt.<br>
   Larger chunks consume more of the context window, potentially limiting the number of different sources that can be referenced.

 - Semantic Search Accuracy :<br>
   Smaller chunks create more granular embeddings, making it easier to find highly specific information that directly answers the question.<br>
   Larger chunks may have broader semantic meaning but could be less precise for specific queries.

 - Information Completeness :<br>
   Smaller chunks might fragment important information that spans across chunk boundaries, leading to incomplete answers.<br>
   Larger chunks are more likely to contain complete information but may include unnecessary details.

 - Processing Speed :<br>
   Smaller chunks generally result in faster embedding generation and retrieval due to their size.
   Larger chunks require more computational resources but may reduce the number of chunks to process.<br>

   After modify the `chunk_size` and `chunk_overlap` it would affect how the system retrieves and processes the information, potentially improving or degrading performance depending on the specific use case.

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Modifying the embedding model can significantly impact on :
 
 - Semantic Understanding Quality:<br>
   Different embedding models have varying capabilities in understanding semantic relationships, context, and nuances in text.<br>
   More advanced models (like text-embedding-3-large vs text-embedding-3-small) can capture more sophisticated semantic patterns and relationships.<br>
   Better semantic understanding leads to more accurate retrieval of relevant context for user queries.

 - Retrieval Accuracy :<br>
   Embedding quality directly affects how well the vector similarity search works.<br>
   Superior embedding models can better distinguish between similar but different concepts, reducing false positives and improving precision.<br>
   Poor embeddings may lead to retrieving irrelevant chunks or missing important context.

 - Multilingual and Domain Performance :<br>
   Different models may perform better on specific languages, domains, or types of content.<br>
   Specialized models might handle technical jargon, industry-specific terms, or cultural context better than general-purpose models.

 - Dimensionality and Representation :<br>
   Different embedding dimensions (e.g., 1536 for text-embedding-3-large vs 1024 for text-embedding-3-small) can capture more or less information.<br>
   Higher dimensionality often means richer representations but may require more computational resources.

 - Training Data and Knowledge Cutoff :<br>
   Newer models may have been trained on more recent data, better understanding current events, terminology, or concepts.<br>
   Model updates can improve performance on specific types of queries or content.

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided, there are several kinds of loans available to students and their parents to help cover the cost of attendance:\n\n1. **Direct Subsidized Loans** – These are loans based on the student's financial need. The government pays the interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These loans are not based on financial need, and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These can be taken out by parents of dependent students to help pay for the student’s cost of attendance, or by independent students themselves if the parent is ineligible. There is no fixed limit on these loans beyond the student's cost of attendance minus other financial aid received.\n\nIt’s also mentioned that students may qualify for a combination of these loans depending on their circumstances, and additional unsubsidized loan amounts are available for independent students

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'cold-church-22' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/b02303d2-7182-4bcf-a2e9-b7c30b9f6ca2/compare?selectedSessions=96a4c8f8-2396-4542-985d-c7f28b5a49cf




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendix A and Appendix B provide guida...,Thank you for your thoughtful question. It’s c...,None,Appendix A offers guidance on the disbursement...,1,0,1,14.011928,c129ff14-f501-40c1-97ff-dd4c38978119,a2a37d74-b28e-4adb-b2c2-14f52e235963
1,"In volume 2 and volume 7, how does the academi...",Thank you for your thoughtful question. I can ...,None,"According to volume 2, the disbursement of fed...",1,1,1,12.382730,200ebfc3-0bc5-41ec-862d-6a8f5be80f7d,903e9f51-65ca-4c27-9e9e-fd0f96404a16
2,whats volume 8 and volume 7 about?,Thank you for your question! Based on the cont...,None,Volume 8 discusses the timing of disbursements...,0,0,1,4.183620,f0d16cb8-1dc8-446d-83e0-87712bc241f2,8a6f99c3-25c1-4903-bfa5-a383e439c163
3,How do Volume 2 and Volume 8 relate to the inc...,Thank you for your thoughtful question. Based ...,None,Volume 2 discusses the requirements for academ...,1,1,1,6.423941,13f56a47-af20-4d4c-918c-befa8470b704,2046679f-338c-48ce-8d29-910a6821894c
4,Hwo do payment perods relate to 34 CFR 668.3(a...,Thank you for your thoughtful question about h...,None,Payment periods are defined within the context...,0,0,1,7.811177,6b508c69-5ff0-46ae-b9c1-6577d43fd2a8,f652cacd-235e-4cdc-8566-4c80d0498ff8
5,How do the disbursement timing requirements fo...,Thank you for your thoughtful question—navigat...,None,Disbursement timing for federal student aid va...,1,0,1,13.754112,9118dee9-e36e-4d56-891a-6b0a943be861,0d45bbd6-a019-4c70-aafa-e30c1a780e5b
6,How do the disbursement timing requirements fo...,Thank you for your thoughtful question. Based ...,None,The disbursement timing requirements for sched...,1,1,1,7.337950,59f97a86-f91a-439b-b41f-ff1b307984d9,ba6abfa4-16ff-482f-bdae-f64e75bafd44
7,How does accelerated progression in clock-hour...,Thank you for your thoughtful question—it show...,None,Accelerated progression in clock-hour or non-t...,1,0,1,10.813890,6de43c54-5e0d-47d7-9ea3-28956d5ceb80,2ccfd6b3-dd9d-4e15-a530-a0f088e87a3e
8,How does Title IV regulation relate to payment...,I understand you're looking for clarity on how...,None,"According to the context, Title IV program dis...",1,1,1,6.404257,c58a4368-d396-4d62-86a9-e2461844f1ad,a83a4207-fd92-41b9-9623-ad1b29e9bd8e
9,What does Volume 8 specify regarding clinical ...,Thank you for your question—it's clear you're ...,None,Volume 8 states that clinical work conducted o...,0,0,1,4.617259,6c937831-dee8-4803-aac2-8e782b81d5fd,8e6e963e-eac8-4268-be13-d886637a0dae


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:
<img src="screenshots/activity3_scrn1.png" alt="Loan Synthetic Data" width="1000"/>
